# How Does Global AI Adoption Rate Compare with Productivity Change? — Phase II
### ENGE707: Data Engineering and Machine Learning Assignment — LCS-Based Prediction

Suemon Kwok (14883335), Quentin Masoe (23191960), Avathanshu Bhat (23227963)

**Dataset:** Global AI Adoption and Workforce Impact Dataset (`ai_company_adoption.csv`)

This notebook continues on from the Phase I scoping report and works through the Phase II tasks (LCS baseline, preprocessing, improved LCS system, experiments, model comparison, interpretation, and discussion).

In [1]:
import os
import zipfile

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

## Task 1 - Project Phase I Summary

In [2]:
# Load the dataset (extract from archive.zip first if the CSV isn't already alongside this notebook)
CSV_NAME = "ai_company_adoption.csv"
ZIP_NAME = "archive.zip"

if not os.path.exists(CSV_NAME) and os.path.exists(ZIP_NAME):
    with zipfile.ZipFile(ZIP_NAME) as z:
        z.extractall(".")

df = pd.read_csv(CSV_NAME)
df.shape

(150000, 43)

### 1.2 Dataset source, size, and characteristics

In [3]:
# Quick structural recap carried over from Phase I
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Distinct companies: {df['company_id'].nunique():,}")
print(f"Countries: {df['country'].nunique()}")
print(f"Industries: {df['industry'].nunique()}")
print(f"Survey period: {df['survey_year'].min()}-{df['quarter'].min()} to {df['survey_year'].max()}-{df['quarter'].max()}")

Rows: 150,000
Columns: 43
Distinct companies: 10,000
Countries: 30
Industries: 9
Survey period: 2023-Q1 to 2026-Q4


In [4]:
# Dtype breakdown
df.dtypes.value_counts()

float64    15
int64      14
str        14
Name: count, dtype: int64

### 1.3 Main data-quality issues identified in Phase I

In [5]:
# Re-confirm the Phase I data-quality findings for the two focus variables
print("Missing values:")
print(df[["ai_adoption_rate", "productivity_change_percent"]].isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate response_id:", df["response_id"].duplicated().sum())

print("\nai_adoption_rate range:", df["ai_adoption_rate"].min(), "-", df["ai_adoption_rate"].max())
print("productivity_change_percent range:", df["productivity_change_percent"].min(), "-", df["productivity_change_percent"].max())

Missing values:
ai_adoption_rate               0
productivity_change_percent    0
dtype: int64



Duplicate rows: 0
Duplicate response_id: 0

ai_adoption_rate range: 0.0 - 100.0
productivity_change_percent range: 0.0 - 34.36


## Task 2 - Original LCS System on Raw Dataset

### 2.1 LCS variant selection

**eLCS**, via the `scikit-eLCS` package (`skeLCS`) — a scikit-learn-compatible implementation of the educational Learning Classifier System (eLCS), exposing the standard `fit` / `predict` / `score` interface.

In [ ]:
from skeLCS import eLCS
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Note: you may need to restart the kernel to use updated packages.


### 2.2 Minimal processing required for LCS compatibility

`skeLCS.eLCS` requires:
- all feature values to be numeric (or `NaN`),
- the target (`phenotype`) to be numeric **and discrete** — this implementation always treats the phenotype as a classification target, so a continuous variable cannot be passed directly.

`ai_adoption_rate` is already numeric, so no change is needed there. `productivity_change_percent`, however, is continuous, so it is minimally processed by binning it into three quantile-based classes (Low / Medium / High) and label-encoding them as integers 0/1/2. This is the smallest possible change that makes the existing Phase I target compatible with the LCS code — no algorithmic modification is made to eLCS itself.

In [7]:
# Minimal processing: discretise the continuous target into 3 balanced classes so it is
# compatible with eLCS's classification-only phenotype requirement
df["productivity_class"] = pd.qcut(
    df["productivity_change_percent"], q=3, labels=[0, 1, 2]
).astype(int)

df["productivity_class"].value_counts().sort_index()

productivity_class
0    50023
1    50031
2    49946
Name: count, dtype: int64

In [8]:
# Baseline feature set: the single Phase I predictor, kept unmodified
X = df[["ai_adoption_rate"]].values.astype(float)
y = df["productivity_class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((120000, 1), (30000, 1))

### 2.3 LCS parameters

Default `skeLCS.eLCS` hyperparameters are used for the baseline (no tuning at this stage — tuning is reserved for the improved system in Task 4):

| Parameter | Value | Meaning |
|---|---|---|
| `learning_iterations` | 10,000 | number of training iterations (rule-learning cycles) |
| `N` | 1,000 | maximum micro-classifier population size |
| `nu` | 5 | fitness pressure exponent |
| `chi` | 0.8 | crossover probability |
| `mu` | 0.04 | mutation probability |
| `p_spec` | 0.5 | probability of specifying an attribute on covering |
| `discrete_attribute_limit` | 10 | max unique values before an attribute is treated as continuous |
| `random_state` | 42 | fixed for reproducibility |

In [9]:
# Train the original, unmodified eLCS baseline
elcs_baseline = eLCS(learning_iterations=10000, random_state=42)
elcs_baseline.fit(X_train, y_train)
print("eLCS baseline trained.")

eLCS baseline trained.


In [10]:
# Baseline performance
train_acc = elcs_baseline.score(X_train, y_train)
test_acc = elcs_baseline.score(X_test, y_test)
coverage = elcs_baseline.get_final_instance_coverage()

print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")
print(f"Final instance coverage: {coverage:.4f}")

Train accuracy: 0.5427
Test accuracy:  0.5399
Final instance coverage: 1.0000


In [11]:
y_pred = elcs_baseline.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["Low", "Medium", "High"]))

              precision    recall  f1-score   support

         Low       0.61      0.72      0.66     10005
      Medium       0.00      0.00      0.00     10006
        High       0.50      0.90      0.64      9989

    accuracy                           0.54     30000
   macro avg       0.37      0.54      0.43     30000
weighted avg       0.37      0.54      0.43     30000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### 2.4 Baseline results summary

Random-guess accuracy for 3 balanced classes is ~33%. The single-feature eLCS baseline should be compared against this floor once results are in — a low margin above chance would support broadening the feature set (or improving preprocessing) as the Task 3/4 improvement, without changing the Task 2 baseline scope itself.